In [25]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import Session, sessionmaker
from contextlib import contextmanager

engine = create_engine("sqlite+pysqlite:///database.db", echo=True)

In [2]:
from sqlalchemy import Column, Integer, String
from sqlalchemy.orm import declarative_base

Base = declarative_base()

class User(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    name = Column(String)
    email = Column(String, unique=True)

    def __repr__(self):
        return f"<User(id={self.id}, name='{self.name}', email='{self.email}')>"

In [3]:
Base.metadata.create_all(engine)

2025-11-07 10:24:35,929 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-07 10:24:35,931 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2025-11-07 10:24:35,931 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-11-07 10:24:35,932 INFO sqlalchemy.engine.Engine COMMIT


In [4]:
with engine.connect() as conn:
    result = conn.execute(text("INSERT INTO users (name, email) VALUES ('John', 'john@example.com')"))
    conn.commit()

2025-11-07 10:24:37,925 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-07 10:24:37,925 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email) VALUES ('John', 'john@example.com')
2025-11-07 10:24:37,926 INFO sqlalchemy.engine.Engine [generated in 0.00155s] ()
2025-11-07 10:24:37,927 INFO sqlalchemy.engine.Engine COMMIT


In [5]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM users"))
    print(result.fetchall())

2025-11-07 10:24:39,642 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-07 10:24:39,643 INFO sqlalchemy.engine.Engine SELECT * FROM users
2025-11-07 10:24:39,643 INFO sqlalchemy.engine.Engine [generated in 0.00100s] ()
[(1, 'John', 'john@example.com')]
2025-11-07 10:24:39,644 INFO sqlalchemy.engine.Engine ROLLBACK


In [13]:
users = Session(engine).query(User)

In [15]:
for user in users:
    print(user)


2025-11-07 10:42:17,620 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email 
FROM users
2025-11-07 10:42:17,622 INFO sqlalchemy.engine.Engine [cached since 753s ago] ()
<User(id=1, name='John', email='john@example.com')>


In [112]:
@contextmanager
def get_db_session():
    with Session(engine) as session:
        yield session
    print("Closing session")


In [113]:
with get_db_session() as session:
    users = session.query(User)
    for user in users:
        print(user)


2025-11-07 11:34:13,902 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-07 11:34:13,903 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email 
FROM users
2025-11-07 11:34:13,903 INFO sqlalchemy.engine.Engine [cached since 2988s ago] ()
<User(id=1, name='John', email='john@example.com')>
2025-11-07 11:34:13,905 INFO sqlalchemy.engine.Engine ROLLBACK
Closing session


In [92]:
class App:
    def __init__(self):
        self.session = get_db_session().__enter__()

    def __enter__(self):
        return self.session

    def __exit__(self, exc_type, exc_value, traceback):
        self.session.close()

In [93]:
with App() as session:
    users = session.query(User)
    for user in users:
        print(user)

2025-11-07 11:16:52,478 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-07 11:16:52,478 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email 
FROM users
2025-11-07 11:16:52,479 INFO sqlalchemy.engine.Engine [cached since 1946s ago] ()
<User(id=1, name='John', email='john@example.com')>
2025-11-07 11:16:52,480 INFO sqlalchemy.engine.Engine ROLLBACK
